# Create zarr storing unstitched tiles using ngff-zarr

Same as `create-tiles.ipynb`, but written with [ngff-zarr](https://github.com/fideus-labs/ngff-zarr) instead of ome-zarr-py.

Related image.sc thread: https://forum.image.sc/t/separate-tiles-to-ome-zarr/109071/60

In [ ]:
from pathlib import Path
import tifffile
import ngff_zarr as nz
from ngff_zarr.v06.zarr_metadata import CoordinateSystem, CoordinateSystemIdentifier, Translation

## Source data

In [ ]:
source_path = Path('../data/example_cells')

filenames = sorted(source_path.glob('*.tiff'))
filenames

## Functions to read tiff

In [ ]:
def read_tiff(filename):
    with tifffile.TiffFile(filename) as tif:
        data = tif.pages.first.asarray()[..., :3]      # drop (constant) alpha channel
    return data.transpose(2, 0, 1)    # yxc -> cyx


def read_tiff_position(filename):
    with tifffile.TiffFile(filename) as tif:
        tags = tif.pages.first.tags
        # XPosition/YPosition hold the aligned tile offset in pixels as a rational (num, denom)
        return {dim: num / denom for dim in 'yx'
                for num, denom in [tags[f'{dim.upper()}Position'].value]}

## Original tile positions

The original (acquisition) tile positions are not stored in the source tiffs, so they are set here from each tile's grid Y/X index, taken from the filename (`<Y>_<X>_<Z>`).

In [ ]:
grid_step = {'y': 820, 'x': 820}     # nominal stage step in pixels: the tile size, i.e. no overlap

original_positions = {}
for filename in filenames:
    grid_y, grid_x = (int(index) for index in filename.stem.split('_')[:2])
    original_positions[filename.stem] = {'y': grid_y * grid_step['y'], 'x': grid_x * grid_step['x']}
original_positions

## Stitched tile positions

The stitched (registered) tile positions are read from the tiffs' XPosition/YPosition metadata.

In [ ]:
stitched_positions = {filename.stem: read_tiff_position(filename) for filename in filenames}
stitched_positions

## Read source images and store as ome-zarr

In [ ]:
dim_order = 'cyx'
pixel_size = {'c': 1, 'y': 0.004, 'x': 0.004}
unit = 'micrometer'
scale_factors = [2, 4]
chunks = (3, 820, 820)
channel_names = ['R', 'G', 'B']
channel_colors = ['FF0000', '00FF00', '0000FF']

output_path = '../output/tiles-ngff.ome.zarr'


images = {}
transforms = []
for filename in filenames:
    data = read_tiff(filename)
    image = nz.to_ngff_image(data, dims=dim_order, scale=pixel_size, name=filename.stem,
                             axes_units={dim: unit for dim in 'yx'}, channel_names=channel_names)
    multiscales = nz.to_multiscales(image, scale_factors=scale_factors, chunks=chunks)
    # to_multiscales doesn't add omero metadata; without it viewers show the channels as grey/default colors
    multiscales.metadata.omero = nz.Omero(channels=[
        nz.OmeroChannel(color=color, window=nz.OmeroWindow(min=0, max=255, start=0, end=255), label=name)
        for name, color in zip(channel_names, channel_colors)])
    stitched_translation = [stitched_positions[filename.stem].get(dim, 0) * pixel_size.get(dim, 1) for dim in dim_order]
    original_translation = [original_positions[filename.stem].get(dim, 0) * pixel_size.get(dim, 1) for dim in dim_order]
    images[filename.stem] = multiscales
    tile_cs = CoordinateSystemIdentifier(path=filename.stem, name=multiscales.metadata.to_version('0.6').coordinateSystems[0].name)
    # stitched: registered tile positions
    transforms.append(Translation(
        translation=stitched_translation,
        input=tile_cs,
        output=CoordinateSystemIdentifier(name='stitched'),
    ))
    # unstitched: original (acquisition) tile positions
    transforms.append(Translation(
        translation=original_translation,
        input=tile_cs,
        output=CoordinateSystemIdentifier(name='unstitched'),
    ))
    print(f'{filename} -> {filename.stem} stitched {stitched_translation}, original {original_translation}')

# reuse the tiles' physical axes for the scene coordinate systems; the first (stitched) is the default view
# (napari-ome-zarr 0.11 instead picks the alphabetically first, so keep stitched first alphabetically too)
first_image = next(iter(images.values()))
axes = first_image.metadata.to_version('0.6').coordinateSystems[0].axes
coordinate_systems = [CoordinateSystem(name=name, axes=axes) for name in ['stitched', 'unstitched']]
print('writing scene')
nz.NgffScene(images=images, coordinateTransformations=transforms, coordinateSystems=coordinate_systems).to_ome_zarr(output_path, version='0.6')
print('done')